# Chương 1 — Lời giải: Đại số tuyến tính

Bản lời giải đã chạy kiểm. Chỉ mở sau khi đã tự làm `lab.ipynb`.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
# Mỗi lab tự tạo Generator riêng (seed cố định) ngay trong ô dữ liệu của nó,
# nên kết quả lab này không phụ thuộc code bạn viết ở lab khác.

## Lab 1 — Power iteration
Tìm λₘₐₓ của ma trận đối xứng bằng cách nhân lặp, đối chiếu với `np.linalg.eigh`.

In [ ]:
def power_iteration(A, n_iter=1000, tol=1e-10):
    v = np.random.default_rng(12345).standard_normal(A.shape[0])   # Generator riêng của hàm
    v /= np.linalg.norm(v)
    lam = 0.0
    for _ in range(n_iter):
        w = A @ v
        v_new = w / np.linalg.norm(w)
        lam_new = v_new @ A @ v_new          # thương Rayleigh
        if abs(lam_new - lam) < tol:
            break
        v, lam = v_new, lam_new
    return lam_new, v_new

In [ ]:
rng = np.random.default_rng(101)
M = rng.standard_normal((50, 50)); S = M @ M.T          # đối xứng, PSD
lam, v = power_iteration(S)
evals, evecs = np.linalg.eigh(S)                         # eigh: thứ tự TĂNG dần
print("power iteration:", lam, "| eigh:", evals[-1])
assert abs(lam - evals[-1]) / evals[-1] < 1e-6, "λ chưa khớp"
assert abs(abs(v @ evecs[:, -1]) - 1) < 1e-6, "vector riêng chưa khớp"
print("Lab 1 ĐẠT ✔")

## Lab 2 — Bình phương tối thiểu ba cách
Đo **sai số nghiệm** ‖x̂ − x‖₂/‖x‖₂, không chỉ phần dư. Dữ liệu và seed nằm trong ô chạy lab; chỉ viết `solve_three_ways`. Câu hỏi: ở 2b bậc 12, `lstsq` cũng sai rất nhiều — lỗi của thuật toán hay của bài toán?

In [ ]:
def solve_three_ways(A, b):
    x_normal = np.linalg.solve(A.T @ A, A.T @ b)      # phương trình chuẩn — không nên dùng
    Q, R = np.linalg.qr(A)
    x_qr = np.linalg.solve(R, Q.T @ b)                # Rx = Qᵀb
    x_lstsq = np.linalg.lstsq(A, b, rcond=None)[0]    # dựa trên SVD
    return {"normal": x_normal, "qr": x_qr, "lstsq": x_lstsq}

In [ ]:
t = np.linspace(0, 1, 50)
results = {}
for deg in [6, 12]:
    rng = np.random.default_rng(200 + deg)               # seed riêng cho từng bậc
    A = np.vander(t, deg + 1, increasing=True)
    x_true = rng.standard_normal(deg + 1)
    b = A @ x_true                                    # (a) hệ nhất quán, không nhiễu
    print(f"bậc {deg}: cond(A) = {np.linalg.cond(A):.1e}")
    for name, x in solve_three_ways(A, b).items():
        fe = np.linalg.norm(x - x_true) / np.linalg.norm(x_true)
        results[(deg, name)] = fe
        print(f"  (a) {name:6s}: sai số nghiệm = {fe:.1e}, phần dư = {np.linalg.norm(A @ x - b):.1e}")
    b_noisy = b + 1e-6 * rng.standard_normal(len(b))  # (b) thêm nhiễu cỡ 1e-6 mỗi phần tử (nhiễu tuyệt đối)
    x = np.linalg.lstsq(A, b_noisy, rcond=None)[0]
    print(f"  (b) lstsq có nhiễu: sai số nghiệm = {np.linalg.norm(x - x_true) / np.linalg.norm(x_true):.1e}")

assert results[(12, "qr")] < 1e-6, "QR phải chính xác"
assert results[(12, "normal")] > 100 * results[(12, "qr")], "Phương trình chuẩn phải kém hơn hẳn ở bậc 12"
print("Lab 2 ĐẠT ✔")

## Lab 3 — PCA bằng SVD
Kiểm công thức sai số của SVD cắt cụt (bình phương sai số tái tạo = tổng σᵢ² bị bỏ) và so với một phép chiếu hạng k ngẫu nhiên.

In [ ]:
def pca_svd(X, k):
    mu = X.mean(axis=0)
    Xc = X - mu                                       # BẮT BUỘC trừ trung bình
    U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Vt[:k], s**2 / (len(X) - 1), mu, s         # thành phần chính = hàng của Vᵀ

In [ ]:
rng = np.random.default_rng(301)
N, d, k = 500, 10, 3
Z = rng.standard_normal((N, k)) * np.array([5.0, 2.0, 1.0])
W = np.linalg.qr(rng.standard_normal((d, d)))[0][:, :k]
X = Z @ W.T + 0.1 * rng.standard_normal((N, d)) + 3.0
comps, var, mu, s = pca_svd(X, k)
X_rec = ((X - mu) @ comps.T) @ comps + mu
err, tail = np.linalg.norm(X - X_rec) ** 2, np.sum(s[k:] ** 2)
print("sai số tái tạo =", err, "| tổng σ² bị bỏ =", tail)
assert np.isclose(err, tail), "công thức sai số chưa khớp"
# So với một phép chiếu hạng k ngẫu nhiên: Eckart–Young nói không phép nào thắng được PCA
Qr = np.linalg.qr(rng.standard_normal((d, k)))[0]
err_rand = np.linalg.norm((X - mu) - (X - mu) @ Qr @ Qr.T) ** 2
print("sai số của chiếu hạng k ngẫu nhiên =", err_rand)
assert err_rand > err, "PCA phải thắng chiếu ngẫu nhiên"
print("Lab 3 ĐẠT ✔")

## Lab 4 — Xấp xỉ hạng thấp: ma trận tổng hợp và ảnh thật
Câu hỏi: vì sao giữ 99% năng lượng lại ứng với sai số tương đối khoảng 10%?

In [ ]:
def lowrank_report(img, ranks):
    U, s, Vt = np.linalg.svd(img, full_matrices=False)
    for r in ranks:
        img_r = (U[:, :r] * s[:r]) @ Vt[:r]
        print(f"  hạng {r:3d}: sai số tương đối {np.linalg.norm(img - img_r) / np.linalg.norm(img):.4f}")
    energy = np.cumsum(s**2) / np.sum(s**2)
    n99 = int(np.searchsorted(energy, 0.99) + 1)
    print("  số thành phần giữ 99% năng lượng:", n99)
    return n99

In [ ]:
rng = np.random.default_rng(401)
synth = np.add.outer(np.sin(np.linspace(0, 6, 200)), np.cos(np.linspace(0, 9, 300)))
synth += 0.05 * rng.standard_normal(synth.shape)
print("4a — ma trận tổng hợp:")
n_syn = lowrank_report(synth, [1, 2, 5, 20])
try:
    from skimage import data
except ImportError:
    data = None
    print("Chưa cài scikit-image (pip install scikit-image): bỏ qua 4b, Lab 4 CHƯA ĐẠT.")
if data is not None:
    print("4b — ảnh thật 512×512:")
    n_cam = lowrank_report(data.camera().astype(float), [1, 5, 20, 50, 100])
    assert n_cam > 5 * n_syn
    print("Lab 4 ĐẠT ✔")

## Săn bug
`pca_buggy` chạy không lỗi nhưng sai. Tìm cả **ba** lỗi rồi viết `pca_fixed`.

In [ ]:
def pca_buggy(X, k):
    C = X.T @ X / (len(X) - 1)
    evals, evecs = np.linalg.eigh(C)
    comps = evecs[:k]
    explained_ratio = evals[:k] / evals.sum()
    return comps, explained_ratio

r1 = np.random.default_rng(1)
Zb = r1.standard_normal((500, 2)) * np.array([5.0, 2.0])
Wb = np.linalg.qr(r1.standard_normal((10, 10)))[0][:, :2]
Xb = Zb @ Wb.T + 0.1 * r1.standard_normal((500, 10)) + 3.0
print("bản lỗi báo tỉ lệ:", pca_buggy(Xb, 2)[1])

In [ ]:
def pca_fixed(X, k):
    Xc = X - X.mean(axis=0)                          # sửa lỗi 1
    C = Xc.T @ Xc / (len(X) - 1)
    evals, evecs = np.linalg.eigh(C)
    order = np.argsort(evals)[::-1]                  # sửa lỗi 2: giảm dần
    evals, evecs = evals[order], evecs[:, order]
    return evecs[:, :k].T, evals[:k] / evals.sum()   # sửa lỗi 3: lấy cột

In [ ]:
comps, ratio = pca_fixed(Xb, 2)
print("bản sửa:", ratio)
cos = np.abs(comps @ Wb)
assert ratio[0] > 0.8 and np.allclose(np.sort(cos.max(axis=1)), 1, atol=0.01)
print("Săn bug ĐẠT ✔")